# Candle Streaks 02b — Does volatility predict UP vs DOWN **after removing zero next returns**?

Independent preregistered binary sensitivity study extending CS02. We keep its **same past-only 1h/24h volatility features**, TRAIN-fitted nine bins, 60/20/20 complete-day chronology and six fitted model hierarchies. We now estimate `P(next UP | immediately next 5m return is NONZERO, current sign, running streak age, volatility level, volatility phase)`. Current zero bars still **end streaks**; a zero next bar is **excluded**, never skipped to a later bar. A comparison additionally normalizes the original TRAIN-fitted CS02 3-class models onto the nonzero outcomes to isolate effects of refitting.

**Interpretation:** Conditioning on a *future* nonzero event is a research diagnostic. A forecaster cannot know that event before the next bar. CS02b does not by itself produce unconditional/actionable `P(up)` or modify AM-T-HAR-Blend. See [CS02b preregistration](../../discussions/Candle_Streaks/002b_chatgpt_binary_direction_volatility_preregistration.md).

## 0 — Environment, source code, isolated unit tests

In [ ]:
import os,sys,json,time,subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from IPython.display import display
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(),'Start JupyterLab inside your cloned repository'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import markov_lift_distributions as vd14
from data_engineering.modeling import candle_streak_volatility as cv
from data_engineering.modeling import candle_streak_volatility_binary as b
cfg=b.Specification()
HIST=Path(os.environ['DERIBIT_DATA_ROOT'])/'history'/'price_5m'
assert HIST.is_dir(),f'Missing Binance 5m history: {HIST}'
OUT=ROOT/'notebooks'/'Candle_Streaks'/'02b';OUT.mkdir(parents=True,exist_ok=True)
sub_env=os.environ.copy();sub_env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+sub_env.get('PYTHONPATH','')
test=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_candle_streak_volatility_binary.py')],
                    cwd=str(ROOT),env=sub_env,text=True,capture_output=True)
print(test.stdout[-1800:]);print(test.stderr[-400:]);assert test.returncode==0
T0=time.monotonic();print('Binary direction hypothesis:',cfg.version)

## 1 — Load the existing five-minute price series and construct causal events

Load BTC/ETH primary and SOL/XRP secondary, when available. Reuse CS02 preprocessing to preserve its streak and volatility feature definitions. Volatility features use returns through the **current** completed candle only. Restrict evaluation to complete UTC days; no target may cross a chronological split. Preserve flat candles as streak breakers, *even though they are not scored as next-bar binary outcomes*.

In [ ]:
raw,excluded=vd14.load_returns(HIST,tuple(cfg.assets),end=pd.Timestamp(cfg.cutoff_utc),min_days=365)
assert {'BTC','ETH'}.issubset(set(raw.underlying.unique())),'BTC/ETH required'
source_end=str(pd.to_datetime(raw.state_time,utc=True).max())
original,quality=cv.build_vol_events(raw,fast_bars=cfg.fast_bars,slow_bars=cfg.slow_bars)
full_days=cv.complete_utc_days(raw)
assets=sorted(original.underlying.unique())
print('Actual last source timestamp:',source_end)
print('Complete days:',{a:len(full_days[a]) for a in assets})
display(quality)

## 2 — EXACTLY the CS02 volatility bins, but binary eligible targets

Freeze volatility cutpoints from all TRAIN-eligible CS02 events **before excluding next-zero labels**. This keeps the volatility partition directly comparable to CS02, with no test-label-driven rebucketing. Then filter only the immediately next zero outcomes from binary fitting and scoring; report precisely how many are excluded.

In [ ]:
prepared={};thresholds={};split_meta={};audits=[]
for asset in assets:
    raw_events=original.loc[original.underlying==asset].copy()
    x,metadata=cv.split_eligible(raw_events,full_days[asset],cfg.train_fraction,cfg.valid_fraction)
    bins=cv.fit_vol_bins(x.loc[x.period=='train'])
    x=cv.apply_vol_bins(x,bins)
    x=b.filter_binary_next(x)
    prepared[asset]=x;thresholds[asset]=bins;split_meta[asset]=metadata
    for period,g in x.groupby('period'):
        audits.append({'asset':asset,'period':period,'observed_current_nonzero':len(g),
                       'scorable_threeclass':int(g.scorable_threeclass.sum()),
                       'excluded_next_zero':int(g.excluded_next_zero.sum()),
                       'binary_next_nonzero_events':int(g.scorable.sum()),
                       'complete_utc_days':int(g.date.nunique())})
eligibility=pd.DataFrame(audits)
display(eligibility)
print('Frozen CS02-compatible bin thresholds:',json.dumps(thresholds,indent=2))

## 3 — Refit binary matrix and compare on equal future origins

The original CS02 six model families are fitted using all TRAIN three-class targets. The analogous binary families refit using only *TRAIN next-up/down outcomes*. Compare both sets on the **same held-out binary targets**. Binary scores: log loss, Brier and class proportion. Use a fixed 50/50 theoretical control; no fitted predictor is allowed to use VALID/TEST outcomes. All actual streak lengths are preserved.

In [ ]:
score_rows=[];bootstrap_rows=[];matrix_rows=[];descriptive_rows=[];monthly_rows=[]
train_params={};binary_fits={};cs02_fits={};max_ages={}
for asset in assets:
    x=prepared[asset];train=x.loc[x.period=='train'].copy()
    train_cs02=train.copy();train_cs02['scorable']=train_cs02.scorable_threeclass
    original_fit=cv.fit_matrix(train_cs02,shrink_sign=cfg.shrink_sign,shrink_streak=cfg.shrink_streak,
                               shrink_sign_vol=cfg.shrink_sign_vol,shrink_joint=cfg.shrink_joint)
    fit=b.fit_binary_matrix(train,shrink_sign=cfg.shrink_sign,shrink_streak=cfg.shrink_streak,
                            shrink_sign_vol=cfg.shrink_sign_vol,shrink_joint=cfg.shrink_joint)
    binary_fits[asset]=fit;cs02_fits[asset]=original_fit
    allkeys=x[['sign','length','level_bin','trend_bin']].drop_duplicates().itertuples(index=False,name=None)
    matrix_rows.append(b.fitted_probability_matrix(fit,include_keys=allkeys).assign(asset=asset))
    descriptive_rows.append(b.descriptive_probability_matrix(x).assign(asset=asset))
    monthly_rows.append(b.monthly_stability(x).assign(asset=asset))
    train_params[asset]={'binary_train_n':fit['n_train'],'cs02_train_n':original_fit['n_train'],
                        'fitted_cells':len(fit['counts']['joint']),
                        'max_k_train':int(train.length.max()),'max_k_all':int(x.length.max()),
                        'binary_unconditional_p_up':float(fit['global'][-1])}
    for period in ('valid','test'):
        eval_=x.loc[(x.period==period)&x.scorable].copy()
        models={**b.predict_binary(eval_,fit),**b.conditionalize_cs02(eval_,original_fit)}
        summary,paired=b.score_binary(eval_,models)
        score_rows.append(summary.assign(asset=asset,period=period))
        bootstrap_rows.append(b.day_bootstrap(paired,B=cfg.boot,seed=cfg.seed).assign(asset=asset,period=period))
scores=pd.concat(score_rows,ignore_index=True)
bootstrap=pd.concat(bootstrap_rows,ignore_index=True)
fitted_matrix=pd.concat(matrix_rows,ignore_index=True)
descriptive=pd.concat(descriptive_rows,ignore_index=True)
monthly=pd.concat(monthly_rows,ignore_index=True)
print('All fits trained strictly on TRAIN-only labels')
display(scores.loc[scores.model.isin(b.MODELS)].pivot_table(index=['asset','period'],columns='model',values='binary_logloss').round(7))
print('Fitted matrix rows:',len(fitted_matrix),'Descriptive matrix rows:',len(descriptive))

## 4 — Visualize fitted probability of next UP (conditional next nonzero)

These are frozen TRAIN estimates, **not** a TEST re-estimate. Display age 1–8 only for readability; every actually observed longer age remains in the saved matrices. Show fitted cell counts to evaluate sparse combinations.

In [ ]:
for asset in assets:
    fig,axs=plt.subplots(1,2,figsize=(12,4),sharey=True)
    for j,sign in enumerate((-1,1)):
        q=fitted_matrix[(fitted_matrix.asset==asset)&(fitted_matrix.sign==sign)&fitted_matrix.length.between(1,8)]
        for lv,tr,label in ((0,0,'low/compression'),(1,1,'mid/neutral'),(2,2,'high/expansion')):
            g=q[(q.level_bin==lv)&(q.trend_bin==tr)].sort_values('length')
            if len(g):axs[j].plot(g.length,100*g.p_up_nonzero,marker='.',label=label)
        axs[j].axhline(50,color='gray',ls='--',alpha=.5)
        axs[j].set(xlabel='Current signed streak age',title=f'{asset}: current sign {sign:+d}')
        axs[j].legend(fontsize=7)
    axs[0].set_ylabel('TRAIN-fitted P(next up | next nonzero), %')
    plt.tight_layout();plt.show()

## 5 — Primary comparisons, complete-day bootstrap and volatility-shuffle null

A directional-volatility claim requires the TRAIN-fitted **sign+vol binary model** to beat sign-only and empirical unconditional binary baselines on BTC and ETH TEST, with negative 95% day-bootstrap upper bounds. The full matrix is a *separate* stronger hypothesis: it must beat streak-only, sign+vol and additive models. Shuffle joint volatility-bin labels within current-sign and coarse-age cohorts as an alignment diagnostic, not a realistic market null.

In [ ]:
test_rows=bootstrap.loc[(bootstrap.period=='test')&(bootstrap.metric=='binary_logloss')]
display(test_rows.loc[test_rows.model.isin(['sign_vol_binary','streak_vol_matrix_binary'])][['asset','model','reference','delta_mean_day','delta_event_weighted','ci025','ci975','n_days']].round(8))
null=[]
for asset in assets:
    te=prepared[asset].loc[lambda g:(g.period=='test')&g.scorable]
    null.append({'asset':asset,**b.shuffle_volatility_null(te,binary_fits[asset],B=cfg.permutation_reps,seed=cfg.seed)})
vol_shuffled=pd.DataFrame(null)
display(vol_shuffled)

## 6 — Frozen decision and aggregate artifacts

Historical TEST has been previously inspected in other projects. Positive results are an exploratory hypothesis, not untouched confirmation or a trade signal. The binary sample **conditions on a future event** and does not assess `P(next zero)`, realized return magnitude, variance, tails, fees or options profitability.

In [ ]:
decisions=[]
for asset in ('BTC','ETH'):
    q=test_rows.loc[test_rows.asset==asset]
    def compare(model,ref):
        x=q[(q.model==model)&(q.reference==ref)].iloc[0]
        return {'reference':ref,'delta':float(x.delta_mean_day),'ci025':float(x.ci025),
                'ci975':float(x.ci975),'passes':bool(x.ci975<0)}
    primary=[compare('sign_vol_binary',ref) for ref in ('sign_binary','iid_binary')]
    matrix=[compare('streak_vol_matrix_binary',ref) for ref in ('streak_binary','sign_vol_binary','additive_binary')]
    decisions.append({'asset':asset,'volatility_adds_binary_directional_information':all(t['passes'] for t in primary),
                      'full_matrix_adds_beyond_simple_models':all(t['passes'] for t in matrix),
                      'volatility_checks':primary,'matrix_checks':matrix})
vol_candidate=all(z['volatility_adds_binary_directional_information'] for z in decisions)
matrix_candidate=all(z['full_matrix_adds_beyond_simple_models'] for z in decisions)
verdict=('EXPLORATORY_BINARY_VOLATILITY_CANDIDATE' if vol_candidate else 'NO_CONFIRMED_BINARY_VOLATILITY_GAIN')
decision={'verdict':verdict,'volatility_binary_candidate':vol_candidate,'full_matrix_binary_candidate':matrix_candidate,
          'primary_assets':decisions,'status':'HISTORICAL_EXPLORATORY','prospective_confirmation':False,
          'actionable_unconditional_next_up_probability':False,'AM_T_HAR_Blend_modified':False}
manifest={'module':'Candle_Streaks_02b','registration':b.registration(),
          'source':'Binance five-minute close-to-close returns via existing VD14 loader',
          'actual_source_last_timestamp':source_end,'excluded':excluded,'quality':quality.to_dict('records'),
          'complete_day_splits':split_meta,'eligibility':eligibility.to_dict('records'),
          'train_fitted_volatility_bins':thresholds,'model_fit':train_params,
          'models':list(b.MODELS),'CS02_threeclass_renormalized_diagnostics':list(b.DIAGNOSTIC_MODELS),
          'decision':decision,'note':'Zero next returns excluded without jumping ahead; current zero breaks streak. Binary forecasts are conditional on future next-bar nonzero.'}
file_outputs={'02b_quality.csv':quality,'02b_eligibility.csv':eligibility,'02b_fit_probability_matrix.csv':fitted_matrix,
              '02b_descriptive_matrix.csv':descriptive,'02b_monthly_stability.csv':monthly,
              '02b_binary_scores.csv':scores,'02b_paired_day_bootstrap.csv':bootstrap,'02b_volatility_shuffle.csv':vol_shuffled}
for file,df in file_outputs.items():df.to_csv(OUT/file,index=False)
for file,obj in {'02b_bins.json':thresholds,'02b_manifest.json':manifest,'02b_decision.json':decision}.items():
    (OUT/file).write_text(json.dumps(obj,indent=2),encoding='utf-8')
print('Final decision:',verdict,'| Full interaction matrix candidate:',matrix_candidate)
print('Output files:',sorted(p.name for p in OUT.iterdir() if p.suffix in ('.csv','.json')))
print('Elapsed seconds:',round(time.monotonic()-T0,2))

## Interpreting CS02b

- All forecasts are for the **immediately next five-minute candle conditional on its return being nonzero**, **NOT** the next nonzero candle at any future time.
- Exact zero-return bars continue to appear in the original time series; they break the running streak. A zero immediately after the current bar is an excluded target, never implicitly removed from the stream or treated as reversal.
- For a deployable two-stage model later, estimate `p0=P(next=0 | known information)` and `q=P(next UP | next!=0, known information)`, then combine `P(up)=(1-p0)q`, `P(down)=(1-p0)(1-q)`.
- This is a five-minute sign study with **observed trailing realized variance**, not the fitted AM-T-HAR-Blend ex-ante Student-t forecast. No existing volatility/density model was changed.
- Compare binary log loss / Brier across independent chronological FIT/VALID/TEST periods. TEST dates were already researched, hence historical exploratory only. Take no trading action on one or two visually striking cells; require stable OOS improvement and calibration before a matched-horizon return-density overlay.